In [75]:
import torch
from torch import nn
from torchrl.envs import PettingZooWrapper
from pettingzooenv import MahjongGameEnv
from torchrl.envs import TransformedEnv
from torchrl.envs.transforms import ActionMask
from torchrl.envs.utils import MarlGroupMapType
from torchrl.modules import MultiAgentConvNet, MultiAgentMLP, ProbabilisticActor, MaskedCategorical
from tensordict.nn import TensorDictModule
from torchrl.collectors import Collector
from torchrl.data.replay_buffers import ReplayBuffer
from torchrl.data.replay_buffers.samplers import SamplerWithoutReplacement
from torchrl.data.replay_buffers.storages import LazyTensorStorage
from torchrl.objectives import ClipPPOLoss, ValueEstimators
# from pygame_visualizer import render_game_state
# import pygame
# import time
# pygame.init()
# screen = pygame.display.set_mode(size=(800, 800))
# font = pygame.font.Font("C:/Windows/Fonts/seguisym.ttf", 48)

env = PettingZooWrapper(env=MahjongGameEnv(), use_mask=True, return_state=True, categorical_actions=True, group_map=MarlGroupMapType.ALL_IN_ONE_GROUP)

# screen.fill('white')
# render_game_state(env._env.gamestate, screen, font)
# pygame.display.update()
# time.sleep(100)

print(env.reset())

TensorDict(
    fields={
        agents: TensorDict(
            fields={
                action_mask: Tensor(shape=torch.Size([4, 75]), device=cpu, dtype=torch.bool, is_shared=False),
                done: Tensor(shape=torch.Size([4, 1]), device=cpu, dtype=torch.bool, is_shared=False),
                mask: Tensor(shape=torch.Size([4]), device=cpu, dtype=torch.bool, is_shared=False),
                observation: TensorDict(
                    fields={
                        observation: Tensor(shape=torch.Size([4, 156, 46]), device=cpu, dtype=torch.uint8, is_shared=False)},
                    batch_size=torch.Size([4]),
                    device=None,
                    is_shared=False),
                terminated: Tensor(shape=torch.Size([4, 1]), device=cpu, dtype=torch.bool, is_shared=False),
                truncated: Tensor(shape=torch.Size([4, 1]), device=cpu, dtype=torch.bool, is_shared=False)},
            batch_size=torch.Size([4]),
            device=None,
            is

In [76]:
device = 'cpu'

In [109]:
class CastToFloat(nn.Module):
    def forward(self, x):
        return x.float()   # or .to(torch.float32)

policy_net = nn.Sequential(
    nn.Flatten(-2),
    CastToFloat(),
    MultiAgentMLP(
        n_agent_inputs = 156 * 46,       
        n_agent_outputs = 75,       
        n_agents = 4,
        centralized = False,        
        share_params = True,      
        depth = 10,               
        num_cells = 1024,
        activation_class=torch.nn.Tanh
    )
)

policy_module = TensorDictModule(
    policy_net,
    in_keys=[("agents", "observation", "observation")],
    out_keys=[("agents", "logits")],
)

policy = ProbabilisticActor(
    module=policy_module,
    spec=env.action_spec_unbatched,
    in_keys={
        'logits': ('agents', 'logits'),
        'mask': ('agents', 'action_mask')
    }, # type: ignore
    out_keys=[env.action_key],
    distribution_class=MaskedCategorical,
    return_log_prob=True
)  # we'll need the log-prob for the PPO loss


In [110]:
critic_net = nn.Sequential(
    nn.Flatten(-2),                    # [248,46] -> [248*46]
    CastToFloat(),
    nn.Linear(248 * 46, 256),
    nn.Tanh(),
    nn.Linear(256, 256),
    nn.Tanh(),
    nn.Linear(256, 256),
    nn.Tanh(),
    nn.Linear(256, 256),
    nn.Tanh(),
    nn.Linear(256, 4),                # output 4 values (one per agent)
    nn.Unflatten(-1, (4, 1))          # reshape from [4] to [4,1]
)

critic = TensorDictModule(
    module=critic_net,
    in_keys=["state"],               # global state
    out_keys=[("agents", "state_value")],  # shape [4] values under agents
).to(device)

In [111]:
print("Running policy:", policy(env.reset()))
print("Running value:", critic(env.reset()))

Running policy: TensorDict(
    fields={
        agents: TensorDict(
            fields={
                action: Tensor(shape=torch.Size([4]), device=cpu, dtype=torch.int64, is_shared=False),
                action_log_prob: Tensor(shape=torch.Size([4]), device=cpu, dtype=torch.float32, is_shared=False),
                action_mask: Tensor(shape=torch.Size([4, 75]), device=cpu, dtype=torch.bool, is_shared=False),
                done: Tensor(shape=torch.Size([4, 1]), device=cpu, dtype=torch.bool, is_shared=False),
                logits: Tensor(shape=torch.Size([4, 75]), device=cpu, dtype=torch.float32, is_shared=False),
                mask: Tensor(shape=torch.Size([4]), device=cpu, dtype=torch.bool, is_shared=False),
                observation: TensorDict(
                    fields={
                        observation: Tensor(shape=torch.Size([4, 156, 46]), device=cpu, dtype=torch.uint8, is_shared=False)},
                    batch_size=torch.Size([4]),
                    device

In [112]:
tensordict_data = env.rollout(max_steps=100, policy=policy)

In [81]:
# from pygame_visualizer import render_game_state
# import pygame
# import time
# pygame.init()
# screen = pygame.display.set_mode(size=(800, 800))
# font = pygame.font.Font("C:/Windows/Fonts/seguisym.ttf", 48)
# from sys import exit
# while True:
#     for event in pygame.event.get():
#         if event.type == pygame.QUIT:
#             pygame.quit()
#             exit()
#     screen.fill('white')
#     render_game_state(env._env.gamestate, screen, font)
#     pygame.display.update()

In [113]:
frames_per_batch = 6_000  # Number of team frames collected per training iteration
n_iters = 5  # Number of sampling and training iterations
total_frames = frames_per_batch * n_iters

collector = Collector(
    env,
    policy,
    device=device,
    storing_device=device,
    frames_per_batch=frames_per_batch,
    total_frames=total_frames,
)
replay_buffer = ReplayBuffer(
    storage=LazyTensorStorage(
        frames_per_batch, device=device
    ),  # We store the frames_per_batch collected at each iteration
    sampler=SamplerWithoutReplacement(),
    batch_size=1,  # We will sample minibatches of this size
)

C:\Users\ctc73\AppData\Local\Temp\ipykernel_34700\1528525382.py:5: FutureWarning: The env passed to Collector is missing transforms required by the policy (InitTracker). From torchrl v0.15 the collector will append them automatically. To enable that behavior now (and silence this warning), pass `auto_register_policy_transforms=True`. To opt out permanently, pass `auto_register_policy_transforms=False`.
  collector = Collector(


In [114]:
loss_module = ClipPPOLoss(
    actor_network=policy,
    critic_network=critic
)
loss_module.set_keys(  # We have to tell the loss where to find the keys
    reward=env.reward_key,
    action=env.action_key,
    value=("agents", "state_value"),
    # These last 2 keys will be expanded to match the reward shape
    done=("agents", "terminated"),                # per-agent
    terminated=("agents", "terminated"),
)
gamma = 0.995  # discount factor
lmbda = 0.9  # lambda for generalised advantage estimation
lr = 3e-4
loss_module.make_value_estimator(
    ValueEstimators.GAE, gamma=gamma, lmbda=lmbda
)  # We build GAE
GAE = loss_module.value_estimator

optim = torch.optim.Adam(loss_module.parameters(), lr)

In [115]:
from tqdm.auto import tqdm
pbar = tqdm(total=n_iters, desc="episode_reward_mean = 0")



episode_reward_mean_list = []
for tensordict_data in collector:
    tensordict_data.set(
        ("next", "agents", "terminated"),
        tensordict_data.get(("next", "terminated"))
        .unsqueeze(-1)
        .expand(tensordict_data.get_item_shape(("next", env.reward_key))),
    )
    # We need to expand the done and terminated to match the reward shape (this is expected by the value estimator)

    with torch.no_grad():
        GAE(
            tensordict_data,
            params=loss_module.critic_network_params,
            target_params=loss_module.target_critic_network_params,
        )  # Compute GAE and add it to the data

    data_view = tensordict_data.reshape(-1)  # Flatten the batch size to shuffle data
    replay_buffer.extend(data_view)

    for _ in range(5):
        for _ in range(frames_per_batch // 1):
            subdata = replay_buffer.sample()
            loss_vals = loss_module(subdata)

            loss_value = (
                loss_vals["loss_objective"]
                + loss_vals["loss_critic"]
                + loss_vals["loss_entropy"]
            )

            loss_value.backward()

            torch.nn.utils.clip_grad_norm_(
                loss_module.parameters(), 1.0
            )  # Optional

            optim.step()
            optim.zero_grad()

    collector.update_policy_weights_()

    # Logging
    done = tensordict_data.get(("next", "terminated"))
    episode_reward_mean = (
        tensordict_data.get(("next", "agents", "episode_reward"))[done].mean().item()
    )
    episode_reward_mean_list.append(episode_reward_mean)
    pbar.set_description(f"episode_reward_mean = {episode_reward_mean}", refresh=False)
    pbar.update()

episode_reward_mean = 0:   0%|          | 0/5 [01:31<?, ?it/s]


2026-07-20 20:55:18,621 [torchrl][INFO]    Initialized LazyTensorStorage with torch.Size([6000]) shape [END]


KeyboardInterrupt: 